<div style="float: block; text-align: center; line-height: 1.7em">
    <span style="font-size: 2em; font-weight: bold"> Fatigue-Sleepiness in Irregular Workloads for Pilots </span><br>
    <span style="font-size: 1.5em; font-weight: bold"> Notebook 3 </span><br>
    <span style="font-size: 1.5em"> ETL </span><br>
</div>

---

---

# 1. Loading Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

---

# 2. Reading files

In [2]:
try:
    file = os.path.join('data/processed_features.csv')
    df = pd.read_csv(file)
    print('-----------------------------------------------')
    print(f"\033[92mSuccess file: {file} read!\033[0m")
    print('-----------------------------------------------')
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success file: data/processed_features.csv read!
-----------------------------------------------


## 2.1. Getting only working days

In [3]:
try:
    df = df[df['duty_moment'].isin(['start','middle','end'])]

    print('-----------------------------------------------')
    print("\033[92mSuccess Working days got!\033[0m")
    print('-----------------------------------------------')
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success Working days got!
-----------------------------------------------


---

# 3. Transforming Variables

## 3.1. Creating labels kssd and spsd

In [4]:
try:
    df['kssd'] = df['kss'].apply(lambda x: 1 if x >= 7 else 0)
    df['spsd'] = df['sps'].apply(lambda x: 1 if x >= 6 else 0)

    #df = df.drop(['kss','sps'], axis = 1)

    print('-----------------------------------------------')
    print(f"\033[92mSuccess labels ksd and spsd created!\033[0m")
    print('-----------------------------------------------')
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success labels ksd and spsd created!
-----------------------------------------------


## 3.2. Time the participant filled the scales

In [5]:
try:
    df['TPFS_slp'] = np.where(df['time_fill_kss_sps'].isin(['EM','EVE']),'em+eve',
                              np.where(df['time_fill_kss_sps'].isin(['MOR','AFT']), 'base', df['time_fill_kss_sps'] ) )
    
    df['TPFS_fat'] = np.where(df['time_fill_kss_sps']=='NI','NI', 'base')

    #df = df.drop(['time_fill_kss_sps'], axis = 1)

    print('-----------------------------------------------')
    print(f"\033[92mSuccess TPFS created!\033[0m")
    print('-----------------------------------------------')
    
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success TPFS created!
-----------------------------------------------


## 3.3. Duty Modality

In [6]:
try:
    df['DMod_slp'] = np.where(df['duty_type']=='night', 'ni','base')
    df['DMod_fat'] = np.where(df['duty_type']=='others', 'base',
                               np.where(df['duty_type']=='early_start', 'es', 
                                        np.where(df['duty_type']=='night', 'ni',df['duty_type'] )))

    print('-----------------------------------------------')
    print(f"\033[92mSuccess DMod created!\033[0m")
    print('-----------------------------------------------')
    
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success DMod created!
-----------------------------------------------


## 3.4. Previous Duty Modalities in the day before Duty

In [7]:
try:
    df['PDMod_fat'] = np.where(df['duty_type_prev_es'] == 1, 'es', 'base')
    #df = df.drop(['duty_type_prev_es','duty_type_prev_nt'], axis = 1)
    
    print('-----------------------------------------------')
    print(f"\033[92mSuccess PDMod created!\033[0m")
    print('-----------------------------------------------')
    
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success PDMod created!
-----------------------------------------------


## 3.5. Circadian Time Conformity

In [8]:
def get_sleep_deviation(x):

    if x.Classification == "MAT":
        if (x.bedtime <= 22.0 and x.bedtime >= 20.0) and x.sleep_duration > 7.0:
            return 0
        else:
            return 1
    elif x.Classification == "INT":
        if (np.floor(x.bedtime) != 0) & (x.bedtime >= 22.0 and x.bedtime <= 23.99) and x.sleep_duration > 7.0:
            return 0
        else:
            return 1
    elif x.Classification == "VES":
        if ((x.bedtime >= 0.0 and x.bedtime <= 1.0) | (x.bedtime >= 23.0 and x.bedtime <= 23.99)) and x.sleep_duration > 7.0:
            return 0
        else:
            return 1
    else:
        return 1

In [9]:
import numpy as np

try:
    df['Bedtime_start_clean'] = df['Bedtime_start'].replace('MISSING', np.nan)
    df = df.dropna(subset = 'Bedtime_start')#.rename(columns = {'Bedtime_start_clean':'Bedtime_start'})
    
    tmp0 = df['Bedtime_start_clean'].str.split(':', expand=True).astype(float).T
    df.loc[:, 'bedtime'] = tmp0.iloc[0] + tmp0.iloc[1] / 60

    df.loc[:, 'sleep_dev'] = df.apply(get_sleep_deviation, axis=1)

    print('-----------------------------------------------')
    print(f"\033[92mSuccess sleep_dev created!\033[0m")
    print('-----------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success sleep_dev created!
-----------------------------------------------


## 3.6. Individuals' Age

In [10]:
try:
    df.loc[:, 'age'] = np.where( (df['Age'] < 35) | (df['Age'] >= 45) , 'base', 'middle' )
    
    print('-----------------------------------------------')
    print(f"\033[92mSuccess variable age engineered!\033[0m")
    print('-----------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success variable age engineered!
-----------------------------------------------


---

# 4. Consolidating Data Frames

In [11]:
# Columns for perception of excessive sleepiness model
cols_slp = ['Id','TPFS_slp','duty_moment','DMod_slp', 'sleep_dev', 'Sleep_quality', 'time_awake', 'sleep_duration', 'age', 'kssd']

# Columns for perception of severe fatigue model
cols_fat = ['Id','TPFS_fat','duty_moment','DMod_fat','PDMod_fat', 'Sleep_quality', 'time_awake', 'duty_length', 'age', 'spsd']

try:
    df_slp_wrk = df.loc[:,cols_slp].copy()
    df_fat_wrk = df.loc[:,cols_fat].copy()

    print('---------------------------------------------------------------')
    print(f"\033[92mSuccess dataframes for sleep and fatigue created!\033[0m")
    print('---------------------------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

---------------------------------------------------------------
Success dataframes for sleep and fatigue created!
---------------------------------------------------------------


## 4.1. Retaining Participants that have more than or equal 10 observations

In [12]:
try:
    id_part_wrk = df_slp_wrk.dropna()[['Id']].groupby('Id', as_index = False).agg(count = ('Id','count'))
    df_slp_wrk = df_slp_wrk.loc[df_slp_wrk['Id'].isin( list(id_part_wrk.loc[id_part_wrk['count']>=10,'Id']) ) ]

    id_part_wrk = df_fat_wrk.dropna()[['Id']].groupby('Id', as_index = False).agg(count = ('Id','count'))
    df_fat_wrk = df_fat_wrk.loc[df_fat_wrk['Id'].isin( list(id_part_wrk.loc[id_part_wrk['count']>=10,'Id']) ) ]

    print('---------------------------------------------------------------')
    print(f"\033[92mSuccess participants with above 10 observations were retained!\033[0m")
    print('---------------------------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

---------------------------------------------------------------
Success participants with above 10 observations were retained!
---------------------------------------------------------------


## 4.2. Saving dataframes

In [13]:
ofile = os.path.join('data','slp_wrk_mdl_train.csv')
try:
    df_slp_wrk.to_csv(ofile, index=False)
    
    print('---------------------------------------------------------------')
    print(f"\033[92mSuccess dataframe saved into {ofile}!\033[0m")
    print('---------------------------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

---------------------------------------------------------------
Success dataframe saved into data\slp_wrk_mdl_train.csv!
---------------------------------------------------------------


In [14]:
ofile = os.path.join('data','fat_wrk_mdl_train.csv')
try:
    df_fat_wrk.to_csv(ofile, index=False)
    
    print('---------------------------------------------------------------')
    print(f"\033[92mSuccess dataframe saved into {ofile}!\033[0m")
    print('---------------------------------------------------------------')

except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

---------------------------------------------------------------
Success dataframe saved into data\fat_wrk_mdl_train.csv!
---------------------------------------------------------------


---